# 01 — Data audit

Validate source scale, timestamps, duplicates, ties and the sample flow before interpretation.

**Executed artifact:** reusable transformations live in `src/` and `sql/`; this notebook reads compact, versioned evidence rather than reprocessing 230 million events interactively.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
def report(name): return json.loads((ROOT/'reports'/name).read_text())

## Source integrity

Exact duplicates and simultaneous events are reported, retained and handled explicitly. Source positions are never presented as behavioral ordering inside a timestamp tie.

In [2]:
a=report('data_audit.json')
pd.DataFrame(a['session_summary'])

,source,sessions,events,clicks,carts,orders,minimum_events,maximum_events,median_events,mean_events,median_duration_hours,mean_duration_hours,mean_repeat_share,sessions_at_least_five
0,test,1671803,13851293,12340303,1155698,355292,2,498,4.0,8.285242,0.210499,14.334356,0.268064,769655
1,train,12899779,216716096,194720954,16896191,5098951,2,500,6.0,16.799985,51.560598,164.594070,0.270400,7624389


In [3]:
pd.DataFrame(a['sample_flow']).T

,source_sessions,sessions_at_least_five,excluded_too_short,eligible_after_time_and_horizon,excluded_time_or_horizon_among_long_enough,target_positive,target_negative,mean_positive_target_set_size,cutoff_tie_extended,click_only_prefixes,mean_prefix_repeat_share
validation,12899779.0,7624389.0,5275390.0,1210610.0,6413779.0,304282.0,906328.0,1.026035,4267.0,914150.0,0.233012
test,1671803.0,769655.0,902148.0,632645.0,137010.0,200864.0,431781.0,1.031046,3372.0,439734.0,0.263928


In [4]:
{k: {x: a['source_ingestion']['sources'][k][x] for x in ['minimum_utc','maximum_utc','duplicate_event_tuples','same_timestamp_adjacent','out_of_order_events']} for k in ['train','test']}

{'train': {'minimum_utc': '2022-07-31T22:00:00.025000+00:00',
  'maximum_utc': '2022-08-28T21:59:59.984000+00:00',
  'duplicate_event_tuples': 331159,
  'same_timestamp_adjacent': 2668780,
  'out_of_order_events': 0},
 'test': {'minimum_utc': '2022-08-28T22:00:00.278000+00:00',
  'maximum_utc': '2022-09-04T21:59:59.984000+00:00',
  'duplicate_event_tuples': 18895,
  'same_timestamp_adjacent': 178112,
  'out_of_order_events': 0}}

## KEY FINDINGS

All official records were processed; exact duplicates and timestamp ties are nonzero and therefore material analytical contracts. The final ranking denominator is explicitly target-positive and smaller than the eligible journey cohort.

## LIMITATIONS

The source cannot reveal which exact duplicates are logging error or real repeated requests. Session boundaries come from OTTO.

## NEXT STEP

Describe observable journeys using timestamp blocks, keeping predictive features separate.